In [12]:
import sys
import os
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm

# Set the project root (one level up from /notebooks/)
project_root = Path(os.getcwd()).parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import your classes and functions directly from the source files
from src.dataset.eyes_dataset import build_dataloaders
from src.models.binary_roi_classifier import build_binary_classifier

In [13]:
STAGE_1_EPOCHS = 10
STAGE_2_EPOCHS = 15
BATCH_SIZE = 64
STAGE_1_LR = 1e-3
STAGE_2_HEAD_LR = 1e-3
STAGE_2_BACKBONE_LR = 1e-4
MIXUP_ALPHA = 0.4
EARLY_STOP_PATIENCE = 5

# Ensure these paths point to your actual data folders
DATASET_ROOTS = {
    "eyes": "../cropped_images/eyes",
    "mouth": "../cropped_images/mouth"
}

CLASS_LABELS = {
    "eyes": {"closed": 0, "open": 1},
    "mouth": {"closed": 0, "open": 1}
}

CHECKPOINT_DIR = Path("runs/binary")

In [14]:
from pathlib import Path

roi = "eyes"
root_path = Path(DATASET_ROOTS[roi])
print(f"Checking root: {root_path.absolute()}")

for class_name in CLASS_LABELS[roi].keys():
    class_dir = root_path / class_name
    images = list(class_dir.glob("*.jpg"))
    print(f"Folder '{class_name}': Found {len(images)} .png images")

Checking root: c:\Users\User\Desktop\University\2026-Autumn\42028-DL-CNN\Assignments\FS\FatigueSense\notebooks\..\cropped_images\eyes
Folder 'closed': Found 319 .png images
Folder 'open': Found 9611 .png images


In [15]:
def mixup_batch(images, labels, alpha):
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    batch_size = images.size(0)
    perm = torch.randperm(batch_size, device=images.device)
    mixed = lam * images + (1 - lam) * images[perm]
    return mixed, labels, labels[perm], lam

def mixup_loss(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    # Adding precision and recall to match your run_stage unpacking
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, precision, recall, f1

In [16]:
def run_stage(model, optimizer, criterion, train_loader, val_loader, device, epochs, start_epoch, use_mixup, checkpoint_path):
    best_val_loss = float("inf")
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "val_f1": []}

    for epoch in range(start_epoch, start_epoch + epochs):
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            if use_mixup:
                mixed, la, lb, lam = mixup_batch(images, labels, MIXUP_ALPHA)
                logits = model(mixed)
                loss = mixup_loss(criterion, logits, la, lb, lam)
            else:
                logits = model(images)
                loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        val_loss, prec, rec, f1 = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(running_loss/len(train_loader.dataset))
        history["val_loss"].append(val_loss)
        history["val_f1"].append(f1)

        print(f"Epoch {epoch:>3} | train_loss={history['train_loss'][-1]:.4f} | val_loss={val_loss:.4f} | F1={f1:.3f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print("Early stop triggered.")
                break
    return history

In [18]:
def train_roi(roi):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training ROI: {roi} on {device}")

    # --- 1. Setup Data ---
    # NOTE: If this fails with num_samples=0, you MUST open src/dataset/eyes_dataset.py
    # and change line 58 from "*.png" to "*.jpg"
    train_loader, val_loader = build_dataloaders(
        root=DATASET_ROOTS[roi],
        class_to_label=CLASS_LABELS[roi],
        batch_size=BATCH_SIZE
    )

    # --- 2. Initialize Model ---
    model = build_binary_classifier(pretrained=True, freeze_backbone=True).to(device)
    criterion = nn.CrossEntropyLoss()
    checkpoint_dir = CHECKPOINT_DIR / roi
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / "best.pt"

    print("\n--- Stage 1: Frozen Backbone ---")
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=STAGE_1_LR)
    run_stage(model, optimizer, criterion, train_loader, val_loader, device, STAGE_1_EPOCHS, 1, False, checkpoint_path)

    print("\n--- Stage 2: Fine-tuning ---")
    model.freeze_backbone(toggle=False, last_n_blocks=2)
    optimizer = torch.optim.Adam([
        {"params": model.backbone.parameters(), "lr": STAGE_2_BACKBONE_LR},
        {"params": model.head.parameters(), "lr": STAGE_2_HEAD_LR},
    ])
    run_stage(model, optimizer, criterion, train_loader, val_loader, device, STAGE_2_EPOCHS, STAGE_1_EPOCHS + 1, True, checkpoint_path)

# Run for Eyes
train_roi("mouth")

Training ROI: mouth on cuda

--- Stage 1: Frozen Backbone ---


Epoch   1 | train_loss=0.0847 | val_loss=0.2141 | F1=0.510


Epoch   2 | train_loss=0.0394 | val_loss=0.0697 | F1=0.719


Epoch   3 | train_loss=0.0319 | val_loss=0.0500 | F1=0.732


Epoch   4 | train_loss=0.0334 | val_loss=0.0539 | F1=0.719


...

Epoch  21 | train_loss=0.0255 | val_loss=0.0349 | F1=0.888


Epoch  22 | train_loss=0.0334 | val_loss=0.0333 | F1=0.888


Epoch  23 | train_loss=0.0279 | val_loss=0.0323 | F1=0.899


Epoch  24 | train_loss=0.0332 | val_loss=0.0316 | F1=0.899


Epoch  25 | train_loss=0.0310 | val_loss=0.0356 | F1=0.899
